# Ejemplo de red bayesiana: probabilidad de aprobar el curso


    HorasEstudio   Asistencia   NotaExamen

          \          |          /

                 Aprobacion

In [1]:
# librerias de redes bayesianas
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

c:\Users\richa\Desktop\UNAMBA\SEMESTRES\SEMESTRE VIII\FUNDAMENTOS DE IA\CODIGO\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\richa\Desktop\UNAMBA\SEMESTRES\SEMESTRE VIII\FUNDAMENTOS DE IA\CODIGO\env\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [2]:
# crear la red bayesiana
model = DiscreteBayesianNetwork([
    ("HorasEstudio", "Aprobacion"),
    ("Asistencia", "Aprobacion"),
    ("NotaExamen", "Aprobacion")
])

## Tablas de probabilidad

In [3]:
# probabilidad previa de las horas de estudio
cpd_horas_estudio = TabularCPD(
    variable="HorasEstudio",
    variable_card=2,
    values=[[0.4], [0.6]],
    state_names={"HorasEstudio": ["Bajas", "Altas"]}
)

# probabilidad previa de la asistencia
cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"Asistencia": ["Baja", "Alta"]}
)

# probabilidad previa de la nota del examen
cpd_nota_examen = TabularCPD(
    variable="NotaExamen",
    variable_card=2,
    values=[[0.35], [0.65]],
    state_names={"NotaExamen": ["Reprobatoria", "Aprobatoria"]}
)

In [4]:
# probabilidad condicional de aprobar según las tres variables de entrada
# Las columnas siguen este orden:
# horas bajas/altas, asistencia baja/alta y nota reprobatoria/aprobatoria.
cpd_aprobacion = TabularCPD(
    variable="Aprobacion",
    variable_card=2,
    values=[
        [0.05, 0.40, 0.20, 0.75, 0.20, 0.70, 0.55, 0.95],
        [0.95, 0.60, 0.80, 0.25, 0.80, 0.30, 0.45, 0.05]
    ],
    evidence=["HorasEstudio", "Asistencia", "NotaExamen"],
    evidence_card=[2, 2, 2],
    state_names={
        "Aprobacion": ["Si", "No"],
        "HorasEstudio": ["Bajas", "Altas"],
        "Asistencia": ["Baja", "Alta"],
        "NotaExamen": ["Reprobatoria", "Aprobatoria"]
    }
)

In [5]:
# agregar al modelo las tablas de probabilidades y verificarlo
model.add_cpds(
    cpd_horas_estudio,
    cpd_asistencia,
    cpd_nota_examen,
    cpd_aprobacion
)
model.check_model()
print("Modelo de red bayesiana creado y verificado correctamente.")
print(model.edges())

Modelo de red bayesiana creado y verificado correctamente.
[('HorasEstudio', 'Aprobacion'), ('Asistencia', 'Aprobacion'), ('NotaExamen', 'Aprobacion')]


In [6]:
# inferencia con los datos del estudiante
inferencia = VariableElimination(model)
resultado_aprobacion = inferencia.query(
    variables=["Aprobacion"],
    evidence={
        "HorasEstudio": "Altas",
        "Asistencia": "Alta",
        "NotaExamen": "Aprobatoria"
    }
)

print("Probabilidad de aprobar con horas de estudio altas, asistencia alta y nota aprobatoria:")
print(resultado_aprobacion)

Probabilidad de aprobar con horas de estudio altas, asistencia alta y nota aprobatoria:
+----------------+-------------------+
| Aprobacion     |   phi(Aprobacion) |
+================+===================+
| Aprobacion(Si) |            0.9500 |
+----------------+-------------------+
| Aprobacion(No) |            0.0500 |
+----------------+-------------------+


In [7]:
# probabilidad de aprobar sin proporcionar evidencia
resultado_aprobacion_sin_evidencia = inferencia.query(
    variables=["Aprobacion"]
)

print("Probabilidad de aprobar sin evidencia:")
print(resultado_aprobacion_sin_evidencia)

Probabilidad de aprobar sin evidencia:
+----------------+-------------------+
| Aprobacion     |   phi(Aprobacion) |
+================+===================+
| Aprobacion(Si) |            0.6241 |
+----------------+-------------------+
| Aprobacion(No) |            0.3759 |
+----------------+-------------------+
